In [8]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from src.data_loading import TARGET_COL, ID_COL, PROCESSED_DATA_PATH, load_data_with_secondary_tables
from src.evaluation import get_feature_importance
from src.tuning import (
    tune_lgbm_fast,
    confirm_lgbm_candidates_cv,
    save_lgbm_params,
)

application_train, application_test = load_data_with_secondary_tables()
X = application_train[[c for c in application_train.columns if c not in [TARGET_COL, ID_COL]]]
y = application_train[TARGET_COL]

X.shape

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


(307511, 596)

In [ ]:
N_TRIALS = 30
TOP_N = 5
TUNING_SAMPLE_SIZE = 100_000
CONFIRM_N_SPLIT = 5
RANDOM_STATE = 42

In [ ]:
tuning_results = tune_lgbm_fast(
    X,
    y,
    n_trials=N_TRIALS,
    top_n=TOP_N,
    sample_size=TUNING_SAMPLE_SIZE,
    random_state=RANDOM_STATE,
)

tuning_results['top_trials']

In [ ]:
candidate_cv_table, candidate_cv_results = confirm_lgbm_candidates_cv(
    X,
    y,
    tuning_results['top_params'],
    n_split=CONFIRM_N_SPLIT,
    random_state=RANDOM_STATE,
)

candidate_summary = tuning_results['top_trials'].merge(
    candidate_cv_table,
    on='candidate',
    how='left',
).sort_values('oof_auc', ascending=False)

candidate_summary

In [ ]:
best_candidate = int(candidate_summary.iloc[0]['candidate'])
best_lgbm_params = tuning_results['top_params'][best_candidate - 1]
best_results = candidate_cv_results[best_candidate - 1]

best_params_path = save_lgbm_params(best_lgbm_params, 'best_lgbm_params.json')
top_params_path = save_lgbm_params(tuning_results['top_params'], 'top_lgbm_params.json')
candidate_summary_path = PROCESSED_DATA_PATH / 'lgbm_tuning_candidate_summary.csv'
oof_path = PROCESSED_DATA_PATH / 'lgbm_engineered_oof.csv'

candidate_summary.to_csv(candidate_summary_path, index=False)
pd.DataFrame({
    ID_COL: application_train[ID_COL],
    TARGET_COL: y,
    'OOF_PRED': best_results['oof_preds'],
}).to_csv(oof_path, index=False)

assert float(candidate_summary.iloc[0]['oof_auc']) >= 0.792

{
    'best_params_path': best_params_path,
    'top_params_path': top_params_path,
    'candidate_summary_path': candidate_summary_path,
    'oof_path': oof_path,
    'best_lgbm_params': best_lgbm_params,
    'fold_aucs': best_results['fold_aucs'],
}

In [ ]:
feature_importance = get_feature_importance(best_results['models'])
feature_importance_path = PROCESSED_DATA_PATH / 'lgbm_engineered_feature_importance.csv'
feature_importance.to_csv(feature_importance_path, index=False)
feature_importance.head(30)